In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F

import re


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 6 (Abstractive Summarization): KNOWLEDGE DISTILLATION
#   TEACHER = Architecture-4 hybrid prefix-LM (205.5M weights, the LARGEST Phase-1 model) + LoRA
#   STUDENT = Architecture-3 E-LegalFormer decoder (125.3M weights) + LoRA, trained with the
#             teacher's next-token probabilities (+ the reference summary) -> submitted
#
# A judgment (and its summary) is much longer than what the models read at once:
#   1. the document is cut into SEGMENTS of about SEGMENT_TOKENS tokens (at sentence ends);
#      500 tokens so that  <eos> segment "Summary:"  fits the bidirectional PREFIX of the teacher
#   2. every sentence of the reference summary is assigned to the segment that shares most
#      of its words and word pairs -> every segment gets its own small target (maybe empty)
#   3. both models learn  <eos> segment "Summary:" -> its part of the summary <eos>
#      (same tokens, same positions: the teacher's and the student's predictions line up)
#   4. STEP 1: the teacher learns from the reference summaries (TEACHER_EPOCHS)
#      STEP 2: the student learns from the reference summaries AND the teacher:
#        loss = (1 - KD_ALPHA) * cross-entropy(reference token)
#               + KD_ALPHA * T^2 * KL(teacher next-token distribution || student)   (every summary token)
#   5. test: the STUDENT summarizes every segment (batched greedy decoding with a KV cache), the
#      parts are joined in DOCUMENT ORDER, repeated sentences are removed, and the summary is
#      trimmed to the expected length (picked on held-out documents)
#
# Data (real Phase-6 files): 7,030 train documents (median 3.1k words = ~7.5k tokens), summaries
# median 640 words (~1.5k tokens), 100 test documents. The summaries copy a lot (78% of their word
# pairs are in the document), cover the WHOLE document in document order, and simply taking the
# first N words of the document already gives ROUGE-1 0.55 (best possible sentence pick: 0.77).
#
# No dev split is given: DEV_DOCS train documents are held out (never trained on).
# Metric here: ROUGE-1 / ROUGE-2 / ROUGE-L F1 (own implementation). The leaderboard's
# BERTScore needs a pretrained model, so it is not computed here: CELL 14 writes dev
# predictions + references for the official evaluation script.
# ============================================================

RUN_NAME = "p6_kd_arch4_to_arch3"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 checkpoints. If a path is wrong, CELL 9 searches /kaggle/input for it.
TEACHER_CHECKPOINT = Path(       # Architecture-4 (HybridLM) best.pt
    "/kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-4/model4_checkpoints/best.pt"
)
STUDENT_CHECKPOINT = Path(       # Architecture-3 (E-LegalFormer) best_model.pt
    "/kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-3/model3_checkpoints/best_model.pt"
)

# Phase-6 data: None = download from Hugging Face (Kaggle secret "HF"),
# or the folder that contains Train/ and Test/, e.g. Path("/kaggle/input/phase-6-summ/Phase_6_Summ")
LOCAL_DATA_DIR = None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Segments
# ------------------------------------------------------------

SEGMENT_TOKENS = 500         # document tokens per segment
TARGET_TOKENS = 384          # longest target (summary part) per segment in training; longer ones are cut
MAX_SEGMENTS = 48            # summarized segments per document (longer documents: 48 spread over it)
EMPTY_KEEP = 0.3             # share of segments with an EMPTY target kept for training (the model learns to skip)
SEPARATOR = "\n\nSummary:\n"


# ------------------------------------------------------------
# Knowledge distillation (Hinton et al., 2015; token level for generation: Kim & Rush, 2016)
# ------------------------------------------------------------

KD_ALPHA = 0.5               # weight of the teacher term (0 = no distillation)
KD_TEMPERATURE = 1.0


# ------------------------------------------------------------
# LoRA (Hu et al., 2021) on every linear layer of every block, for teacher AND student
#   W' = W + alpha/r * B A ; Phase-1 weights (embeddings, tied LM head) stay FROZEN
# ------------------------------------------------------------

TEACHER_TARGETS = ["encoder_attention.q_proj", "encoder_attention.k_proj", "encoder_attention.v_proj",
                   "encoder_attention.out_proj", "causal_attention.q_proj", "causal_attention.k_proj",
                   "causal_attention.v_proj", "causal_attention.out_proj", "hybrid_fusion.gate_proj",
                   "hybrid_fusion.out_proj", "ffn.gate_proj", "ffn.up_proj", "ffn.down_proj"]
STUDENT_TARGETS = ["attention.q_proj", "attention.k_proj", "attention.v_proj", "attention.o_proj",
                   "ffn.gate_proj", "ffn.up_proj", "ffn.down_proj"]
PEFT_RANK = 16
PEFT_ALPHA = 32
PEFT_DROPOUT = 0.05

LR_PEFT = 2e-4


# ------------------------------------------------------------
# Training (TRAIN documents only, minus the held-out ones)
# ------------------------------------------------------------

TEACHER_EPOCHS = 1           # teacher: reference summaries only
STUDENT_EPOCHS = 2           # student: reference summaries + teacher (the teacher runs alongside, no gradients)
EPOCH_EXAMPLES = 16000       # (segment, target) examples per epoch (all used before any repeats)
BATCH_SIZE = 4               # sequences per forward pass (each <= ~1.7k tokens). If CUDA out-of-memory: 2
GRAD_ACCUM = 4               # -> 16 sequences per weight update
WARMUP_RATIO = 0.03
GRAD_CLIP = 1.0

DEV_DOCS = 300               # train documents held out: dev loss after every epoch (picks the best epoch)
DEV_LOSS_EXAMPLES = 1000     # dev examples for the dev loss
GEN_DEV_DOCS = 60            # held-out documents summarized at the end: pick the length + ROUGE report

# Skip STEP 1: the teacher_trainable.pt saved by an earlier run of THIS notebook. None = train the teacher.
TEACHER_TRAINED_PATH = None


# ------------------------------------------------------------
# Generation (greedy, KV cache)
# ------------------------------------------------------------

GEN_BATCH = 24               # segments decoded together. If CUDA out-of-memory: 12
MAX_NEW_TOKENS = 384         # per segment (hard limit)
PART_BUDGET = 1.5            # a segment may generate up to 1.5 x its share of the expected summary
                             # (tokens); the summary is trimmed afterwards. Much faster than always 512
NO_REPEAT_NGRAM = 4          # a sequence of 4 non-space tokens (~3 words) is never generated twice in a part
                             # (8 let short loops through: "court court court ...")
EOS_PENALTIES = [0.0, 2.0, 4.0]   # subtracted from the <eos> score: > 0 = longer parts (greedy decoding tends to stop
                             # early); both are tried on dev, the summary is then trimmed to the best length
LENGTH_SCALES = [0.6, 0.8, 1.0, 1.25, None]     # x the expected summary length (None = no trimming), picked on dev


# ------------------------------------------------------------
# Debug mode (quick run on a few documents)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 120

if DEBUG_MODE:
    EPOCHS = 1
    EPOCH_EXAMPLES = 64
    DEV_DOCS = 20
    DEV_LOSS_EXAMPLES = 32
    GEN_DEV_DOCS = 8


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


print("Run       :", RUN_NAME)
print("Segments  :", f"{SEGMENT_TOKENS} document tokens -> up to {TARGET_TOKENS} summary tokens")
print("KD        :", f"teacher Architecture-4 -> student Architecture-3 | alpha {KD_ALPHA}, T {KD_TEMPERATURE}")
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")


Run       : p6_kd_arch4_to_arch3
Segments  : 500 document tokens -> up to 384 summary tokens
KD        : teacher Architecture-4 -> student Architecture-3 | alpha 0.5, T 1.0
Output    : /kaggle/working/p6_kd_arch4_to_arch3
AMP dtype : torch.float16


In [3]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER + FAST ENCODING
#
# Documents are long, so only the parts we need are tokenized,
# and every pre-token piece is BPE-encoded only once (cache).
# Placeholders (<ENTITY>, <SECTION>, <ACT>) are replaced first.
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


piece_cache = {}      # pre-token piece -> token ids


def encode_fast(text):
    """Same ids as tokenizer.encode(text), but every piece is encoded only once."""

    ids = []

    for piece in tok_module.PRETOKEN_PATTERN.findall(tok_module.normalize_text(text)):

        if piece not in piece_cache:
            piece_cache[piece] = tokenizer.encode(piece)

        ids.extend(piece_cache[piece])

    return ids


EOS_ID = tokenizer.eos_id
BOS_ID = tokenizer.bos_id
SEPARATOR_IDS = encode_fast(SEPARATOR)

# whitespace-only tokens (" ", "\n", ...): this tokenizer makes every space its own token
def token_text(i):
    try:
        return tokenizer.id_to_token[i]
    except (KeyError, IndexError):
        return None


SPACE_TOKEN_IDS = {i for i in range(VOCAB_SIZE)
                   if i != PAD_ID and isinstance(token_text(i), str) and token_text(i).strip() == ""}


def decode(ids):
    return tokenizer.decode([int(i) for i in ids]).strip()


demo = "The appeal is allowed. The order of the High Court dated 12.03.2010 is set aside."
assert encode_fast(demo) == tokenizer.encode(demo)
assert EOS_ID is not None

print(f"EOS {EOS_ID} | BOS {BOS_ID} | separator {SEPARATOR!r} -> {len(SEPARATOR_IDS)} tokens | "
      f"whitespace-only tokens: {len(SPACE_TOKEN_IDS)}")
print("decode(encode(text)) == text:", decode(encode_fast(demo)) == demo)


Vocabulary size: 32004 | PAD / UNK: 0 1
EOS 3 | BOS 2 | separator '\n\nSummary:\n' -> 5 tokens | whitespace-only tokens: 3
decode(encode(text)) == text: True


In [4]:
# ============================================================
# CELL 4 — PHASE-6 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE6_FILES = {
    "train": "Phase_6_Summ/Train/train.json",
    "test":  "Phase_6_Summ/Test/test.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE6_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_6_Summ/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE6_FILES.items():
        DATA_FILES[name] = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                           repo_type="dataset", token=HF_TOKEN)

for name, path in DATA_FILES.items():
    print(f"{name:6s} -> {path}  ({os.path.getsize(path) / 1e6:.0f} MB)")


train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Train/train.json  (211 MB)
test   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Test/test.json  (3 MB)


In [5]:
# ============================================================
# CELL 5 — LOAD DOCUMENTS + SUMMARIES, HOLD OUT DEV DOCUMENTS, EXPECTED SUMMARY LENGTH
# ============================================================

def read_json(path):
    """A JSON file (list, or {id: ...} object), or a JSON-lines file."""

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        data = [json.loads(line) for line in content.splitlines() if line.strip()]

    if isinstance(data, dict):                 # {"doc_001": {...}} or {"doc_001": "text"}
        data = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "document": v} for k, v in data.items()]

    return data


def load_docs(path, with_summary=True):

    docs = []

    for i, item in enumerate(read_json(path)):
        docs.append({
            "id": str(item.get("id", i)),
            "document": str(item.get("document", item.get("text", "")) or ""),
            "summary": str(item.get("summary", "") or "") if with_summary else None,
        })

    if DEBUG_MODE:
        docs = docs[:DEBUG_DOCS]

    return docs


all_train_docs = load_docs(DATA_FILES["train"])
_n = len(all_train_docs)
all_train_docs = [d for d in all_train_docs if d["document"].strip() and d["summary"].strip()]
print(f"train pairs with an empty document or summary dropped: {_n - len(all_train_docs)}")
test_docs = load_docs(DATA_FILES["test"], with_summary=False)          # no reference summaries: never read

ids = [d["id"] for d in test_docs]
assert len(ids) == len(set(ids)), "test ids are not unique"

# hold out DEV_DOCS train documents (no dev split is given)
_order = list(range(len(all_train_docs)))
random.Random(SEED).shuffle(_order)
dev_docs = [all_train_docs[i] for i in _order[:DEV_DOCS]]
train_docs = [all_train_docs[i] for i in _order[DEV_DOCS:]]


def word_count(text):
    return len(text.split())


print(f"{'split':6s} {'docs':>6s} {'document words (median / 90% / max)':>38s} {'summary words (median / 90%)':>30s}")
for name, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    dw = np.array([word_count(d["document"]) for d in docs])
    line = f"{name:6s} {len(docs):6d} {np.median(dw):14,.0f} / {np.percentile(dw, 90):7,.0f} / {dw.max():9,.0f}"
    if docs[0]["summary"] is not None:
        sw = np.array([word_count(d["summary"]) for d in docs])
        line += f" {np.median(sw):14,.0f} / {np.percentile(sw, 90):7,.0f}"
    print(line)

sample = random.Random(SEED + 1).sample(train_docs, min(200, len(train_docs)))
tokens_per_word = (sum(len(encode_fast(d["document"])) for d in sample[:50]) /
                   max(1, sum(word_count(d["document"]) for d in sample[:50])))
TOKENS_PER_WORD = tokens_per_word
print(f"\ntokens per word (Phase-1 tokenizer): {tokens_per_word:.2f}")

placeholders = Counter(m for d in sample for m in re.findall(r"<[A-Z_]+>", d["document"] + d["summary"]))
print("placeholders like <ENTITY> (kept as they are):", dict(placeholders.most_common(5)) or "none")


# ------------------------------------------------------------
# Expected summary length: log(summary words) = a + b * log(document words), fitted on train
# ------------------------------------------------------------

_dw = np.log([max(word_count(d["document"]), 1) for d in train_docs])
_sw = np.log([max(word_count(d["summary"]), 1) for d in train_docs])
LENGTH_B, LENGTH_A = np.polyfit(_dw, _sw, 1)
_r2 = 1 - np.var(_sw - (LENGTH_A + LENGTH_B * _dw)) / np.var(_sw)
SUMMARY_WORDS_MIN, SUMMARY_WORDS_MAX = np.percentile(np.exp(_sw), [2, 98])


def expected_summary_words(document):
    words = np.exp(LENGTH_A + LENGTH_B * np.log(max(word_count(document), 1)))
    return float(np.clip(words, SUMMARY_WORDS_MIN, SUMMARY_WORDS_MAX))


print(f"summary words = {np.exp(LENGTH_A):.2f} x document words ^ {LENGTH_B:.2f}   (R^2 {_r2:.2f} on log scale)")
print(f"\nexample summary (start): {train_docs[0]['summary'][:300]!r}")


train pairs with an empty document or summary dropped: 2
split    docs    document words (median / 90% / max)   summary words (median / 90%)
train    6728          3,129 /   7,840 /   139,645            638 /   1,495
dev       300          3,132 /   8,255 /    90,410            626 /   1,626
test      100          3,171 /   8,563 /    39,695

tokens per word (Phase-1 tokenizer): 2.41
placeholders like <ENTITY> (kept as they are): none
summary words = 4.59 x document words ^ 0.61   (R^2 0.45 on log scale)



In [6]:
# ============================================================
# CELL 6 — SENTENCES -> SEGMENTS -> WHICH SUMMARY SENTENCES BELONG TO WHICH SEGMENT
# ============================================================

ABBREVIATIONS = {"v", "vs", "no", "nos", "s", "ss", "sec", "secs", "art", "arts", "dr", "mr", "mrs", "ms", "smt",
                 "sri", "shri", "co", "ltd", "pvt", "etc", "viz", "j", "jj", "cj", "para", "paras", "p", "pp",
                 "cr", "crl", "rs", "u", "ors", "anr", "dt", "st", "govt", "ch", "cl", "sub", "ibid", "supra",
                 "hon'ble", "honble", "i.e", "e.g", "w.e.f", "a.i.r", "s.c.r", "s.c.c", "r.s.a", "o.s", "c.a"}

_BOUNDARY = re.compile(r'(?<=[.?!])\s+(?=["\'(\[]?[A-Z0-9])')


def split_sentences(text):
    """Sentences of a legal text (line breaks are boundaries; common legal abbreviations are not)."""

    sentences = []

    for paragraph in re.split(r"\n+", text):

        start = 0
        for m in _BOUNDARY.finditer(paragraph):

            words = paragraph[start:m.start()].split()
            last = words[-1].lower().rstrip(".?!") if words else ""

            if last in ABBREVIATIONS or len(last) <= 1:
                continue                                            # e.g. "v.", "No.", "S.", initials

            sentences.append(paragraph[start:m.start()].strip())
            start = m.end()

        if paragraph[start:].strip():
            sentences.append(paragraph[start:].strip())

    return [s for s in sentences if s]


def words_of(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def make_segments(document):
    """Sentences packed into segments of <= SEGMENT_TOKENS tokens -> list of {"ids", "text"}."""

    segments, ids, texts = [], [], []

    def close():
        if ids:
            segments.append({"ids": np.array(ids, dtype=np.int32), "text": " ".join(texts)})

    for sentence in split_sentences(document):

        sentence_ids = encode_fast(sentence if not ids else " " + sentence)

        if ids and len(ids) + len(sentence_ids) > SEGMENT_TOKENS:     # does not fit: start a new segment
            close()
            ids, texts = [], []
            sentence_ids = encode_fast(sentence)

        if len(sentence_ids) > SEGMENT_TOKENS:                      # a very long "sentence": cut into pieces
            for j in range(0, len(sentence_ids), SEGMENT_TOKENS):
                piece = sentence_ids[j:j + SEGMENT_TOKENS]
                segments.append({"ids": np.array(piece, dtype=np.int32), "text": decode(piece)})
            continue

        ids += sentence_ids
        texts.append(sentence)

    close()

    return segments or [{"ids": np.array([EOS_ID], dtype=np.int32), "text": ""}]


def bigrams(words):
    return set(zip(words, words[1:]))


def align_summary(segments, summary):
    """Every summary sentence -> the segment with the highest unigram + 2 x bigram recall.
    Returns (one target text per segment, mean bigram recall of the assigned sentences)."""

    seg_words = [set(words_of(s["text"])) for s in segments]
    seg_bigrams = [bigrams(words_of(s["text"])) for s in segments]

    parts = [[] for _ in segments]
    recalls = []

    for sentence in split_sentences(summary):

        w = words_of(sentence)
        if not w:
            continue
        b = bigrams(w)
        uw = set(w)

        scores = [len(uw & sw) / len(uw) + 2 * (len(b & sb) / len(b) if b else 0.0)
                  for sw, sb in zip(seg_words, seg_bigrams)]
        best = int(np.argmax(scores))

        parts[best].append(sentence)
        recalls.append(len(b & seg_bigrams[best]) / len(b) if b else 1.0)

    return [" ".join(p) for p in parts], float(np.mean(recalls)) if recalls else 0.0


def spread(segments):
    """At most MAX_SEGMENTS segments, spread over the whole document."""

    if len(segments) <= MAX_SEGMENTS:
        return segments
    keep = np.unique(np.linspace(0, len(segments) - 1, MAX_SEGMENTS).round().astype(int))
    return [segments[i] for i in keep]


t0 = time.time()

for name, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:

    for k, d in enumerate(docs):

        d["segments"] = make_segments(d["document"])
        d["words"] = word_count(d["document"])

        if d["summary"] is not None:
            d["targets"], d["align_recall"] = align_summary(d["segments"], d["summary"])

        if (k + 1) % 2000 == 0:
            print(f"  {name}: {k + 1}/{len(docs)} documents | {time.time() - t0:.0f}s", flush=True)

print(f"Segmented + aligned in {time.time() - t0:.0f}s")


# ------------------------------------------------------------
# Training examples: (segment ids, target ids); most EMPTY targets are dropped
# ------------------------------------------------------------

def make_examples(docs, rng):

    examples, n_empty, n_cut = [], 0, 0

    for d in docs:
        for segment, target in zip(d["segments"], d["targets"]):

            target_ids = encode_fast(target) if target else []

            if not target_ids:
                n_empty += 1
                if rng.random() > EMPTY_KEEP:
                    continue

            if len(target_ids) > TARGET_TOKENS:
                n_cut += 1
                target_ids = target_ids[:TARGET_TOKENS]

            examples.append((segment["ids"], np.array(target_ids, dtype=np.int32)))

    return examples, n_empty, n_cut


train_examples, n_empty, n_cut = make_examples(train_docs, random.Random(SEED))
dev_examples, _, _ = make_examples(dev_docs, random.Random(SEED + 1))
dev_examples = random.Random(SEED).sample(dev_examples, min(DEV_LOSS_EXAMPLES, len(dev_examples)))

n_segments = [len(d["segments"]) for d in train_docs]
target_lengths = [len(t) for _, t in train_examples]

print(f"\nsegments per train document: median {np.median(n_segments):.0f}, 90% {np.percentile(n_segments, 90):.0f}, max {max(n_segments)}")
print(f"summary sentences found in their segment (bigram recall): {np.mean([d['align_recall'] for d in train_docs]):.2f}")
print(f"train examples: {len(train_examples):,} (empty targets: {n_empty:,}, {EMPTY_KEEP:.0%} of them kept) | "
      f"target tokens median {np.median(target_lengths):.0f}, 90% {np.percentile(target_lengths, 90):.0f} "
      f"| cut at {TARGET_TOKENS}: {n_cut}")
print(f"dev examples (dev loss): {len(dev_examples):,}")


  train: 2000/6728 documents | 32s
  train: 4000/6728 documents | 65s
  train: 6000/6728 documents | 96s
Segmented + aligned in 113s

segments per train document: median 18, 90% 44, max 785
summary sentences found in their segment (bigram recall): 0.64
train examples: 105,843 (empty targets: 86,229, 30% of them kept) | target tokens median 99, 90% 312 | cut at 384: 5075
dev examples (dev loss): 1,000


In [7]:
# ============================================================
# CELL 7 — ROUGE-1 / ROUGE-2 / ROUGE-L F1 (lower-case words; LCS with a fast bit-parallel algorithm)
# ============================================================

def ngram_f1(reference, candidate, n):

    ref = Counter(zip(*[reference[i:] for i in range(n)]))
    cand = Counter(zip(*[candidate[i:] for i in range(n)]))
    overlap = sum((ref & cand).values())

    if overlap == 0:
        return 0.0

    precision, recall = overlap / sum(cand.values()), overlap / sum(ref.values())
    return 2 * precision * recall / (precision + recall)


def lcs_length(a, b):
    """Length of the longest common subsequence (bit-parallel, Allison-Dix / Hyyro)."""

    if not a or not b:
        return 0

    masks = {}
    for i, token in enumerate(a):
        masks[token] = masks.get(token, 0) | (1 << i)

    full = (1 << len(a)) - 1
    v = full
    for token in b:
        u = v & masks.get(token, 0)
        v = ((v + u) | (v - u)) & full

    return len(a) - bin(v).count("1")


def rouge_l_f1(reference, candidate):

    lcs = lcs_length(reference, candidate)
    if lcs == 0:
        return 0.0

    precision, recall = lcs / len(candidate), lcs / len(reference)
    return 2 * precision * recall / (precision + recall)


def rouge(references, candidates):
    """Mean ROUGE-1 / ROUGE-2 / ROUGE-L F1 over documents + mean candidate length (words)."""

    r1, r2, rl = [], [], []

    for reference, candidate in zip(references, candidates):
        ref, cand = words_of(reference), words_of(candidate)
        r1.append(ngram_f1(ref, cand, 1))
        r2.append(ngram_f1(ref, cand, 2))
        rl.append(rouge_l_f1(ref, cand))

    return {"rouge1": float(np.mean(r1)), "rouge2": float(np.mean(r2)), "rougeL": float(np.mean(rl)),
            "words": float(np.mean([word_count(c) for c in candidates]))}


# check the LCS against plain dynamic programming
def _lcs_dp(a, b):
    row = [0] * (len(b) + 1)
    for x in a:
        prev = 0
        for j, y in enumerate(b, 1):
            prev, row[j] = row[j], prev + 1 if x == y else max(row[j], row[j - 1])
    return row[-1]

_rng = random.Random(0)
for _ in range(200):
    _a = [_rng.randint(0, 5) for _ in range(_rng.randint(0, 40))]
    _b = [_rng.randint(0, 5) for _ in range(_rng.randint(0, 40))]
    assert lcs_length(_a, _b) == _lcs_dp(_a, _b)
print("✅ ROUGE check passed")


✅ ROUGE check passed


In [8]:
# ============================================================
# CELL 8 — MODELS: TEACHER (Architecture-4) + STUDENT (Architecture-3), LoRA, distillation loss,
#          batched generation of the student with a KV cache
# ============================================================

# ============================================================
# Architecture-4 HYBRID PREFIX-LM (layer names as in the Phase-1 notebook)
#   every block: encoder branch (bidirectional attention over the PREFIX) + causal branch,
#   mixed by a learned gate. Tokens after the prefix get 0 from the encoder branch (as in
#   pretraining, prefix_length 512).
#   Here: prefix = <eos> segment "Summary:" (<= 512 tokens), then the summary part (causal).
# ============================================================

class HybridRMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (self.weight * (xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps))).to(x.dtype)


class HybridAttention(nn.Module):

    def __init__(self, d_model, n_heads):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None, causal=False, cache=None):

        B, T, D = x.shape
        q, k, v = (p(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
                   for p in (self.q_proj, self.k_proj, self.v_proj))

        if cache is not None:
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v

        if mask is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)

        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, D))


class GatedHybridFusion(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        self.gate_proj = nn.Linear(2 * d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, encoder_output, causal_output):
        gate = torch.sigmoid(self.gate_proj(torch.cat([encoder_output, causal_output], dim=-1)))
        return self.out_proj(gate * encoder_output + (1.0 - gate) * causal_output)


class HybridSwiGLU(nn.Module):

    def __init__(self, d_model, ffn_dim):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class HybridTransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        d, h, eps = cfg["d_model"], cfg["n_heads"], cfg["norm_eps"]
        self.encoder_norm = HybridRMSNorm(d, eps)
        self.causal_norm = HybridRMSNorm(d, eps)
        self.encoder_attention = HybridAttention(d, h)
        self.causal_attention = HybridAttention(d, h)
        self.hybrid_fusion = GatedHybridFusion(d)
        self.ffn_norm = HybridRMSNorm(d, eps)
        self.ffn = HybridSwiGLU(d, cfg["ffn_dim"])

    def forward(self, x, prefix_mask=None, causal_mask=None, cache=None):
        """prefix_mask [B, P]: the first P positions of x hold the prefix (True = prefix token);
        None = no prefix in x (generation steps after the prompt)."""

        if prefix_mask is not None:
            P = prefix_mask.size(1)
            encoder_output = self.encoder_attention(self.encoder_norm(x[:, :P]), mask=prefix_mask[:, None, None, :])
            encoder_output = encoder_output * prefix_mask[..., None].to(encoder_output.dtype)
            encoder_output = F.pad(encoder_output, (0, 0, 0, x.size(1) - P))
        else:
            encoder_output = torch.zeros_like(x)

        causal_output = self.causal_attention(self.causal_norm(x), mask=causal_mask,
                                              causal=causal_mask is None, cache=cache)

        x = x + self.hybrid_fusion(encoder_output, causal_output)
        return x + self.ffn(self.ffn_norm(x))


class Arch4LM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["block_size"], cfg["d_model"])
        self.blocks = nn.ModuleList([HybridTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = HybridRMSNorm(cfg["d_model"], cfg["norm_eps"])
        self.lm_head = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.token_embedding.weight                  # weight tying (as in Phase 1)

    def forward(self, ids, positions=None, prefix_mask=None, causal_mask=None, caches=None):

        if positions is None:
            positions = torch.arange(ids.size(1), device=ids.device)

        x = self.token_embedding(ids) + self.position_embedding(positions)
        for i, block in enumerate(self.blocks):
            x = block(x, prefix_mask, causal_mask, None if caches is None else caches[i])

        return self.final_norm(x)


# ============================================================
# Architecture-3 E-LegalFormer DECODER (GQA: 12 query heads share 4 key/value heads, RoPE)
# Layer names as in the Phase-1 notebook. Generation: prompts LEFT-padded, own positions per row.
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_float = x.float()
        x_float = x_float * torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * self.weight).to(x.dtype)


class RotaryEmbedding(nn.Module):

    def __init__(self, dim, max_seq_len, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        freqs = torch.outer(torch.arange(max_seq_len), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)


def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    """cos / sin: [T, dim/2] (same positions for every row) or [B, T, dim/2] (own positions per row)."""
    cos = torch.cat([cos, cos], dim=-1)
    sin = torch.cat([sin, sin], dim=-1)
    cos = (cos[None, None] if cos.dim() == 2 else cos[:, None]).to(x.dtype)
    sin = (sin[None, None] if sin.dim() == 2 else sin[:, None]).to(x.dtype)
    return x * cos + rotate_half(x) * sin


class GroupedQueryAttention(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.n_q_heads = cfg["n_q_heads"]
        self.n_kv_heads = cfg["n_kv_heads"]
        self.head_dim = cfg["d_model"] // cfg["n_q_heads"]
        self.q_proj = nn.Linear(cfg["d_model"], self.n_q_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg["d_model"], self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg["d_model"], self.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg["d_model"], cfg["d_model"], bias=False)

    def forward(self, x, cos, sin, mask=None, cache=None):

        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

        if cache is not None:                                  # the cache keeps the 4 key/value heads
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v

        repeat = self.n_q_heads // self.n_kv_heads
        k = k.repeat_interleave(repeat, dim=1)
        v = v.repeat_interleave(repeat, dim=1)

        if mask is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)

        return self.o_proj(out.transpose(1, 2).contiguous().view(B, T, C))


class SwiGLU3(nn.Module):

    def __init__(self, d_model, ffn_dim):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class ELegalFormerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attention = GroupedQueryAttention(cfg)
        self.ffn_norm = RMSNorm(cfg["d_model"])
        self.ffn = SwiGLU3(cfg["d_model"], cfg["ffn_dim"])

    def forward(self, x, cos, sin, mask=None, cache=None):
        x = x + self.attention(self.attn_norm(x), cos, sin, mask, cache)
        return x + self.ffn(self.ffn_norm(x))


class Arch3LM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.rope = RotaryEmbedding(cfg["d_model"] // cfg["n_q_heads"], cfg["max_seq_len"], cfg.get("rope_theta", 10000.0))
        self.layers = nn.ModuleList([ELegalFormerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.token_embedding.weight                  # weight tying (as in Phase 1)

    def forward(self, ids, positions=None, mask=None, caches=None):

        if positions is None:
            cos, sin = self.rope.cos[:ids.size(1)], self.rope.sin[:ids.size(1)]
        else:
            cos, sin = self.rope.cos[positions], self.rope.sin[positions]

        x = self.token_embedding(ids)
        for i, layer in enumerate(self.layers):
            x = layer(x, cos, sin, mask, None if caches is None else caches[i])

        return self.final_norm(x)


# ============================================================
# Training batches (same for both): <eos> segment SEPARATOR | target <eos>
# ============================================================

def make_prompt(segment_ids):
    return [EOS_ID] + segment_ids.tolist() + SEPARATOR_IDS


def make_batch(examples):
    """-> ids [B, T] (right padding), loss_mask [B, T] (True where the NEXT token is a target token),
    prefix_lengths [B] (the teacher's bidirectional prefix)."""

    sequences, starts = [], []
    for segment_ids, target_ids in examples:
        prompt = make_prompt(segment_ids)
        sequences.append(prompt + target_ids.tolist() + [EOS_ID])
        starts.append(len(prompt) - 1)

    T = max(len(s) for s in sequences)
    ids = torch.full((len(sequences), T), PAD_ID, dtype=torch.long)
    loss_mask = torch.zeros((len(sequences), T), dtype=torch.bool)

    for row, (s, start) in enumerate(zip(sequences, starts)):
        ids[row, :len(s)] = torch.tensor(s)
        loss_mask[row, start:len(s) - 1] = True

    prefix_lengths = torch.tensor([start + 1 for start in starts])
    return ids.to(DEVICE), loss_mask.to(DEVICE), prefix_lengths.to(DEVICE)


def token_logits(model, examples):
    """Logits of every target token (+ the final <eos>) -> (logits [N, V] float32, targets [N]).
    Teacher and student: the same N positions in the same order."""

    ids, loss_mask, prefix_lengths = make_batch(examples)

    with autocast():
        if isinstance(model, Arch4LM):                    # teacher: prefix = <eos> segment separator
            P = int(prefix_lengths.max())
            hidden = model(ids, prefix_mask=torch.arange(P, device=DEVICE)[None] < prefix_lengths[:, None])
        else:                                             # student: causal decoder
            hidden = model(ids)
        logits = model.lm_head(hidden[:, :-1][loss_mask[:, :-1]])

    return logits.float(), ids[:, 1:][loss_mask[:, :-1]]


def lm_loss(model, examples):
    logits, targets = token_logits(model, examples)
    return F.cross_entropy(logits, targets)


def kd_loss(student, teacher, examples):
    """(1 - KD_ALPHA) * CE(reference) + KD_ALPHA * T^2 * KL(teacher || student), every summary token."""

    student_logits, targets = token_logits(student, examples)
    with torch.no_grad():
        teacher_logits, teacher_targets = token_logits(teacher, examples)
    assert torch.equal(targets, teacher_targets)

    T = KD_TEMPERATURE
    ce = F.cross_entropy(student_logits, targets)
    kl = F.kl_div(F.log_softmax(student_logits / T, dim=-1), F.log_softmax(teacher_logits / T, dim=-1),
                  log_target=True, reduction="batchmean") * T * T

    return (1 - KD_ALPHA) * ce + KD_ALPHA * kl


# ============================================================
# LoRA (teacher and student)
# ============================================================

class LoRALinear(nn.Module):

    def __init__(self, base, rank, alpha, dropout):
        super().__init__()

        self.base = base                                      # frozen Phase-1 layer
        self.lora_A = nn.Linear(base.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, base.out_features, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale = alpha / rank

        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)                    # starts with no change

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_B(self.lora_A(self.dropout(x)))


def add_lora(model, blocks, targets):
    """Freeze every Phase-1 weight, then wrap the target linear layers of every block with LoRA."""

    for p in model.parameters():
        p.requires_grad = False

    for block in blocks:
        for target in targets:
            parent_name, name = target.rsplit(".", 1)
            parent = block.get_submodule(parent_name)
            setattr(parent, name, LoRALinear(getattr(parent, name), PEFT_RANK, PEFT_ALPHA, PEFT_DROPOUT))


# ============================================================
# Greedy choice of the next token (shared by all models)
#   - never PAD / UNK / BOS
#   - eos_penalty is subtracted from the <eos> score (> 0 = longer parts)
#   - no sequence of NO_REPEAT_NGRAM non-space tokens is generated twice in a part
#   - a row stops at <eos> or after budgets[row] tokens
# ============================================================

BANNED_IDS = [i for i in (PAD_ID, UNK_ID, BOS_ID) if i is not None]


class GreedyState:
    """Bookkeeping of the rows of one generation batch."""

    def __init__(self, budgets):
        self.budgets = budgets
        self.outputs = [[] for _ in budgets]
        self.content = [[] for _ in budgets]          # generated non-space tokens
        self.seen = [dict() for _ in budgets]         # (n-1 tokens) -> next tokens seen
        self.finished = [False] * len(budgets)


def pick_next_tokens(logits, state, eos_penalty):
    """logits [B, V] (float32) -> next token of every row (greedy) + bookkeeping."""

    logits[:, BANNED_IDS] = -float("inf")
    logits[:, EOS_ID] -= eos_penalty

    if NO_REPEAT_NGRAM:
        for row, c in enumerate(state.content):
            if not state.finished[row] and len(c) >= NO_REPEAT_NGRAM - 1:
                banned = state.seen[row].get(tuple(c[-(NO_REPEAT_NGRAM - 1):]))
                if banned:
                    logits[row, list(banned)] = -float("inf")

    next_tokens = logits.argmax(dim=-1)

    for row, token in enumerate(next_tokens.tolist()):

        if state.finished[row]:
            continue
        if token == EOS_ID:
            state.finished[row] = True
            continue

        state.outputs[row].append(token)
        if len(state.outputs[row]) >= state.budgets[row]:
            state.finished[row] = True

        if token not in SPACE_TOKEN_IDS:
            c = state.content[row]
            c.append(token)
            if NO_REPEAT_NGRAM and len(c) >= NO_REPEAT_NGRAM:
                state.seen[row].setdefault(tuple(c[-NO_REPEAT_NGRAM:-1]), set()).add(token)

    return next_tokens


# ============================================================
# Batched greedy generation of the STUDENT (Architecture-3) with a KV cache
# ============================================================

@torch.no_grad()
def generate(model, prompts, budgets, eos_penalty=0.0):
    """List of prompt id lists -> list of generated id lists."""

    model.eval()

    B, P = len(prompts), max(len(p) for p in prompts)
    ids = torch.full((B, P), PAD_ID, dtype=torch.long)
    key_mask = torch.zeros((B, P), dtype=torch.bool)
    for row, p in enumerate(prompts):
        ids[row, P - len(p):] = torch.tensor(p)
        key_mask[row, P - len(p):] = True
    ids, key_mask = ids.to(DEVICE), key_mask.to(DEVICE)
    positions = (key_mask.long().cumsum(dim=1) - 1).clamp(min=0)

    causal = torch.ones(P, P, dtype=torch.bool, device=DEVICE).tril()
    mask = (causal[None] & key_mask[:, None, :]) | torch.eye(P, dtype=torch.bool, device=DEVICE)[None]

    caches = [{} for _ in model.layers]
    with autocast():
        hidden = model(ids, positions, mask[:, None], caches)
        logits = model.lm_head(hidden[:, -1]).float()

    last_position = positions[:, -1]
    state = GreedyState(budgets)

    for step in range(max(budgets)):

        next_tokens = pick_next_tokens(logits, state, eos_penalty)
        if all(state.finished):
            break

        key_mask = torch.cat([key_mask, torch.ones((B, 1), dtype=torch.bool, device=DEVICE)], dim=1)
        last_position = last_position + 1

        with autocast():
            hidden = model(next_tokens[:, None], last_position[:, None], key_mask[:, None, None, :], caches)
            logits = model.lm_head(hidden[:, -1]).float()

    model.train()
    return state.outputs


def plain_next_token(model, prompt, generated):
    """Next token by a full forward pass (no cache) - only used to check generate()."""

    model.eval()
    with torch.no_grad(), autocast():
        logits = model.lm_head(model(torch.tensor([prompt + generated], device=DEVICE))[:, -1]).float()
    logits[:, BANNED_IDS] = -float("inf")
    return int(logits.argmax())


In [9]:
# ============================================================
# CELL 9 — LOAD BOTH PHASE-1 CHECKPOINTS + BUILDERS
# ============================================================

def find_checkpoint(path, file_name, required_key):
    """Use `path` if it exists, else search /kaggle/input for `file_name` whose config has `required_key`."""

    if Path(path).exists():
        return Path(path)

    for candidate in sorted(Path("/kaggle/input").rglob(file_name)):
        try:
            ckpt = torch.load(candidate, map_location="cpu", weights_only=False)
            if required_key in ckpt.get("model_config", ckpt.get("config", {})):
                print("Found:", candidate)
                return candidate
        except Exception:
            pass

    raise FileNotFoundError(f"{path} not found and no matching {file_name} under /kaggle/input")


def load_weights(path, config_key, tied_key):
    phase1 = torch.load(path, map_location="cpu", weights_only=False)
    config = dict(phase1[config_key])
    weights = {key.replace("_orig_mod.", ""): value for key, value in phase1["model_state_dict"].items()}
    if "lm_head.weight" not in weights:                   # tied: saved once
        weights["lm_head.weight"] = weights[tied_key]
    assert torch.equal(weights["lm_head.weight"], weights[tied_key]), "expected a tied LM head"
    return config, weights


TEACHER_CHECKPOINT = find_checkpoint(TEACHER_CHECKPOINT, "best.pt", "prefix_length")
TEACHER_CONFIG, TEACHER_WEIGHTS = load_weights(TEACHER_CHECKPOINT, "model_config", "token_embedding.weight")
TEACHER_CONFIG.setdefault("norm_eps", 1e-5)

STUDENT_CHECKPOINT = find_checkpoint(STUDENT_CHECKPOINT, "best_model.pt", "n_kv_heads")
STUDENT_CONFIG, STUDENT_WEIGHTS = load_weights(STUDENT_CHECKPOINT, "config", "token_embedding.weight")

print("Teacher config:", TEACHER_CONFIG)
print("Student config:", STUDENT_CONFIG)

for config in (TEACHER_CONFIG, STUDENT_CONFIG):
    assert config["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert 1 + SEGMENT_TOKENS + len(SEPARATOR_IDS) + max(TARGET_TOKENS, MAX_NEW_TOKENS) + 1 <= min(
    TEACHER_CONFIG["block_size"], STUDENT_CONFIG["max_seq_len"])
if 1 + SEGMENT_TOKENS + len(SEPARATOR_IDS) > TEACHER_CONFIG["prefix_length"]:
    print(f"NOTE: the teacher prefix is longer than in its pretraining ({TEACHER_CONFIG['prefix_length']})")


def build(model, weights, blocks_of, targets):
    """Phase-1 model + LoRA (checks: nothing changes at the start, only LoRA trains)."""
    model.to(DEVICE)                                  # ADD

    model.load_state_dict(weights, strict=True)
    test_example = [(np.arange(10, 40, dtype=np.int32), np.arange(40, 52, dtype=np.int32))]

    model.eval()
    with torch.no_grad():
        before, _ = token_logits(model, test_example)
        add_lora(model, blocks_of(model), targets)
        model.to(DEVICE) 
        after, _ = token_logits(model, test_example)
    model.train()

    assert torch.allclose(before, after, atol=1e-3), "LoRA changed the model at the start"
    names = [name for name, p in model.named_parameters() if p.requires_grad]
    assert names and all("lora_" in name for name in names), "only the LoRA matrices may train"

    return model.to(DEVICE)


def build_teacher():
    return build(Arch4LM(TEACHER_CONFIG), TEACHER_WEIGHTS, lambda m: m.blocks, TEACHER_TARGETS)


def build_student():
    return build(Arch3LM(STUDENT_CONFIG), STUDENT_WEIGHTS, lambda m: m.layers, STUDENT_TARGETS)


TEACHER_SIZE = sum(v.numel() for k, v in TEACHER_WEIGHTS.items() if k != "lm_head.weight")
STUDENT_SIZE = sum(v.numel() for k, v in STUDENT_WEIGHTS.items() if k != "lm_head.weight")
print(f"\nTeacher: {TEACHER_SIZE:,} weights | student: {STUDENT_SIZE:,} weights ({TEACHER_SIZE / STUDENT_SIZE:.2f}x smaller)")


Teacher config: {'vocab_size': 32004, 'block_size': 2048, 'd_model': 768, 'n_layers': 16, 'n_heads': 12, 'ffn_dim': 2048, 'prefix_length': 512, 'dropout': 0.0, 'norm_eps': 1e-05, 'bias': False}
Student config: {'vocab_size': 32004, 'max_seq_len': 2048, 'd_model': 768, 'n_layers': 16, 'n_q_heads': 12, 'n_kv_heads': 4, 'ffn_dim': 2048, 'dropout': 0.0, 'rope_theta': 10000.0}

Teacher: 205,496,064 weights | student: 125,267,712 weights (1.64x smaller)


In [10]:
# ============================================================
# CELL 10 — SUMMARIZE DOCUMENTS
#   every segment -> generated part (batched; segments with similar budget + length together);
#   budget of a segment = PART_BUDGET x its share of the expected summary length (tokens)
#   parts in document order -> sentences; repeated sentences removed -> trimmed to the
#   expected length x scale (keeps the first sentences of every part)
# ============================================================

@torch.no_grad()
def generate_parts(model, docs, eos_penalty):
    """doc["parts"] = generated text of each of its segments (spread to MAX_SEGMENTS)."""

    jobs = []
    for d_index, d in enumerate(docs):

        segments = spread(d["segments"])
        share = expected_summary_words(d["document"]) * TOKENS_PER_WORD / len(segments)
        budget = int(np.clip(PART_BUDGET * share, 48, MAX_NEW_TOKENS))

        for s_index, segment in enumerate(segments):
            jobs.append((d_index, s_index, make_prompt(segment["ids"]), budget))
        d["parts"] = [""] * len(segments)

    jobs.sort(key=lambda job: (job[3], len(job[2])))
    t0 = time.time()

    for k in range(0, len(jobs), GEN_BATCH):

        batch = jobs[k:k + GEN_BATCH]
        outputs = generate(model, [job[2] for job in batch], [job[3] for job in batch], eos_penalty)

        for (d_index, s_index, _, _), out in zip(batch, outputs):
            docs[d_index]["parts"][s_index] = decode(out)

        if (k // GEN_BATCH + 1) % 20 == 0:
            print(f"    {min(k + GEN_BATCH, len(jobs))}/{len(jobs)} segments | {time.time() - t0:.0f}s", flush=True)

    return len(jobs), time.time() - t0


def assemble(doc, scale):
    """Join the generated parts into one summary of about scale x the expected length."""

    kept, kept_bigrams = [], []                    # (part index, position in part, sentence)

    for p_index, part in enumerate(doc["parts"]):
        for position, sentence in enumerate(split_sentences(part)):

            b = bigrams(words_of(sentence))
            if b and any(len(b & other) / len(b) > 0.7 for other in kept_bigrams):
                continue                                                   # repeated sentence

            kept.append((p_index, position, sentence))
            kept_bigrams.append(b)

    if scale is None or not kept:
        return " ".join(s for _, _, s in kept)

    target = scale * expected_summary_words(doc["document"])
    total = sum(word_count(s) for _, _, s in kept)

    if total > target:                                                     # keep the first sentences of every part
        ratio = target / total
        per_part = Counter(p for p, _, _ in kept)
        kept = [(p, i, s) for p, i, s in kept if i < max(1, math.ceil(per_part[p] * ratio))]

    words = " ".join(s for _, _, s in kept).split()
    return " ".join(words[:int(math.ceil(target))]) if len(words) > target * 1.1 else " ".join(words)


In [11]:
# ============================================================
# CELL 11 — TRAINING FUNCTION (teacher or student)
#   LoRA matrices only (Phase-1 weights frozen); loss on the target tokens (student: + the teacher);
#   after every epoch: loss on held-out dev examples -> the best epoch is kept
# ============================================================

@torch.no_grad()
def dev_loss(model, examples):

    model.eval()
    total, count = 0.0, 0
    for k in range(0, len(examples), BATCH_SIZE * 2):
        batch = examples[k:k + BATCH_SIZE * 2]
        total += lm_loss(model, batch).item() * len(batch)
        count += len(batch)
    model.train()

    return total / max(1, count)


def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}


def batches_by_length(examples, rng):
    """Shuffle, then group examples of similar length (less padding) into batches of BATCH_SIZE."""

    examples = examples[:]
    rng.shuffle(examples)

    batches = []
    for k in range(0, len(examples), BATCH_SIZE * 32):
        pool = sorted(examples[k:k + BATCH_SIZE * 32], key=lambda e: len(e[0]) + len(e[1]))
        batches += [pool[j:j + BATCH_SIZE] for j in range(0, len(pool), BATCH_SIZE)]

    rng.shuffle(batches)
    return batches


def train_model(model, epochs, teacher=None):
    """teacher None: plain LM loss (the teacher itself). teacher given: distillation loss (the student)."""

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR_PEFT, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    per_epoch = min(EPOCH_EXAMPLES, len(train_examples))
    total_steps = epochs * math.ceil(per_epoch / (BATCH_SIZE * GRAD_ACCUM))
    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    def set_lr(step):
        factor = min(1.0, (step + 1) / warmup_steps) * max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps)) \
            if step >= warmup_steps else (step + 1) / warmup_steps
        optimizer.param_groups[0]["lr"] = LR_PEFT * factor

    rng = random.Random(SEED)
    order = train_examples[:]
    rng.shuffle(order)
    cursor = 0

    history, step, t_start = [], 0, time.time()

    loss0 = dev_loss(model, dev_examples)
    best_loss, best_epoch, best_state = loss0, 0, trainable_state(model)
    history.append({"epoch": 0, "dev_loss": loss0})
    print(f"Epoch 0 (Phase-1 model, no training) | dev loss {loss0:.4f} (cross-entropy of the reference tokens)", flush=True)

    model.train()

    for epoch in range(1, epochs + 1):

        if cursor + per_epoch > len(order):
            rng.shuffle(order)
            cursor = 0

        batches = batches_by_length(order[cursor:cursor + per_epoch], rng)
        cursor += per_epoch

        total_loss, skipped = 0.0, 0
        optimizer.zero_grad(set_to_none=True)

        for b, batch in enumerate(batches):

            loss = lm_loss(model, batch) if teacher is None else kd_loss(model, teacher, batch)
            scaler.scale(loss / GRAD_ACCUM).backward()
            total_loss += loss.item()

            if (b + 1) % GRAD_ACCUM == 0 or b + 1 == len(batches):

                set_lr(step)
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)

                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped += scaler.get_scale() < scale_before

                optimizer.zero_grad(set_to_none=True)
                step += 1

            if (b + 1) % 500 == 0:
                print(f"    batch {b + 1}/{len(batches)} | loss {total_loss / (b + 1):.4f} | "
                      f"{(time.time() - t_start) / 60:.1f} min", flush=True)

        loss_dev = dev_loss(model, dev_examples)
        history.append({"epoch": epoch, "train_loss": total_loss / len(batches), "dev_loss": loss_dev, "skipped_steps": int(skipped)})

        line = (f"Epoch {epoch}/{epochs} | train loss {total_loss / len(batches):.4f} | dev loss {loss_dev:.4f} "
                f"(perplexity {math.exp(min(loss_dev, 20)):.1f})")
        if loss_dev < best_loss:
            best_loss, best_epoch, best_state = loss_dev, epoch, trainable_state(model)
            line += "  <- best"
        print(line + f" | {(time.time() - t_start) / 60:.1f} min", flush=True)

    model.load_state_dict(best_state, strict=False)                    # back to the best epoch
    return history, best_epoch


In [12]:
# ============================================================
# CELL 12 — STEP 1: TEACHER (Architecture-4 + LoRA) on the reference summaries
#           STEP 2: STUDENT (Architecture-3 + LoRA) with knowledge distillation
#   dev loss = cross-entropy of the reference summary tokens of held-out documents (both models)
# ============================================================

for name in ["teacher", "model", "optimizer", "scaler"]:
    globals().pop(name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


# ---------------- STEP 1: teacher ----------------
teacher = build_teacher()
print(f"STEP 1 | teacher LoRA weights: {sum(p.numel() for p in teacher.parameters() if p.requires_grad):,}")

t0 = time.time()
if TEACHER_TRAINED_PATH is None:
    teacher_history, teacher_best_epoch = train_model(teacher, TEACHER_EPOCHS)
else:
    teacher.load_state_dict(torch.load(TEACHER_TRAINED_PATH, map_location="cpu"), strict=False)
    teacher_history, teacher_best_epoch = [], "loaded"
teacher_minutes = (time.time() - t0) / 60

torch.save(trainable_state(teacher), OUTPUT_DIR / "teacher_trainable.pt")
for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()

teacher_dev_loss = dev_loss(teacher, dev_examples)
teacher.eval()
print(f"\n✅ Teacher: dev loss {teacher_dev_loss:.4f} | {teacher_minutes:.1f} min")


# ---------------- STEP 2: student with the teacher ----------------
model = build_student()                     # `model` = the student (dev report + test summaries)
n_total = sum(p.numel() for p in model.parameters())
n_trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nSTEP 2 | student LoRA weights: {n_trained:,} | KD alpha {KD_ALPHA}, T {KD_TEMPERATURE}")

t0 = time.time()
history, best_epoch = train_model(model, STUDENT_EPOCHS, teacher=teacher)
train_minutes = (time.time() - t0) / 60
peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(model), OUTPUT_DIR / "best_trainable.pt")
student_dev_loss = dev_loss(model, dev_examples)

# the teacher is not needed any more
del teacher
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print(f"\n✅ Student trained in {train_minutes:.1f} min | best epoch {best_epoch} | dev loss {student_dev_loss:.4f} "
      f"(teacher {teacher_dev_loss:.4f}) | peak GPU {peak_gpu_gb:.2f} GB")


# ------------------------------------------------------------
# Check: student generation with the KV cache = plain forward passes
# ------------------------------------------------------------

_prompts = [make_prompt(np.array(encode_fast("The appeal was filed against the order of the High Court")[:12], dtype=np.int32)),
            make_prompt(np.array(encode_fast("The respondent")[:5], dtype=np.int32))]
_saved, NO_REPEAT_NGRAM = NO_REPEAT_NGRAM, 0
_greedy = generate(model, _prompts, [6, 6])
NO_REPEAT_NGRAM = _saved
for _p, _g in zip(_prompts, _greedy):
    for _k, _t in enumerate(_g):
        assert plain_next_token(model, _p, _g[:_k]) == _t, "KV-cache generation differs from the plain forward pass"
model.train()
print("✅ KV-cache generation = plain forward passes")


STEP 1 | teacher LoRA weights: 6,291,456
Epoch 0 (Phase-1 model, no training) | dev loss 2.0096 (cross-entropy of the reference tokens)
    batch 500/4000 | loss 2.0818 | 3.2 min
    batch 1000/4000 | loss 1.9418 | 6.0 min
    batch 1500/4000 | loss 1.8922 | 8.8 min
    batch 2000/4000 | loss 1.8611 | 11.6 min
    batch 2500/4000 | loss 1.8389 | 14.3 min
    batch 3000/4000 | loss 1.8227 | 17.0 min
    batch 3500/4000 | loss 1.8139 | 19.8 min
    batch 4000/4000 | loss 1.8055 | 22.5 min
Epoch 1/1 | train loss 1.8055 | dev loss 1.7969 (perplexity 6.0)  <- best | 23.0 min

✅ Teacher: dev loss 1.7969 | 23.2 min

STEP 2 | student LoRA weights: 3,473,408 | KD alpha 0.5, T 1.0
Epoch 0 (Phase-1 model, no training) | dev loss 1.5011 (cross-entropy of the reference tokens)
    batch 500/4000 | loss 1.1113 | 3.2 min
    batch 1000/4000 | loss 0.9715 | 6.0 min
    batch 1500/4000 | loss 0.9167 | 8.9 min
    batch 2000/4000 | loss 0.8833 | 11.7 min
    batch 2500/4000 | loss 0.8608 | 14.5 min
    

In [13]:
# ============================================================
# CELL 13 — DEV REPORT (the STUDENT, held-out train documents) + TEACHER vs STUDENT
#   summaries of GEN_DEV_DOCS documents -> the best length scale (ROUGE-1 + ROUGE-L)
#   references: LEAD = the first words of the document (same length), and
#   ALIGNED = the reference sentences regrouped by segment (shows what the segment
#   assignment keeps: an upper bound for this approach)
# ============================================================

gen_dev_docs = dev_docs[:GEN_DEV_DOCS]
references = [d["summary"] for d in gen_dev_docs]

print(f"{'summary':34s} {'ROUGE-1':>8s} {'ROUGE-2':>8s} {'ROUGE-L':>8s} {'words':>7s}")
print(f"{'reference':34s} {'':>8s} {'':>8s} {'':>8s} {np.mean([word_count(r) for r in references]):7.0f}")

scale_results, best = {}, None
for penalty in EOS_PENALTIES:

    n_segments_dev, gen_seconds = generate_parts(model, gen_dev_docs, penalty)
    parts = {d["id"]: d["parts"] for d in gen_dev_docs}

    for scale in LENGTH_SCALES:
        candidates = [assemble(d, scale) for d in gen_dev_docs]
        scores = rouge(references, candidates)
        scale_results[f"eos_penalty {penalty}, length x {scale}"] = scores
        print(f"{f'model, eos -{penalty}, length x {scale}':34s} {scores['rouge1']:8.4f} {scores['rouge2']:8.4f} "
              f"{scores['rougeL']:8.4f} {scores['words']:7.0f}")

        if best is None or scores["rouge1"] + scores["rougeL"] > best[0]["rouge1"] + best[0]["rougeL"]:
            best = (scores, penalty, scale, parts)

    print(f"    ({n_segments_dev} segments in {gen_seconds:.0f}s = {n_segments_dev / max(gen_seconds, 1e-9):.1f} segments/s)")

DEV_SCORES, BEST_PENALTY, BEST_SCALE, _parts = best
for d in gen_dev_docs:
    d["parts"] = _parts[d["id"]]

lead = [" ".join(d["document"].split()[:int(expected_summary_words(d["document"]))]) for d in gen_dev_docs]
LEAD_SCORES = rouge(references, lead)
aligned = [" ".join(t for t in d["targets"] if t) for d in gen_dev_docs]
ALIGNED_SCORES = rouge(references, aligned)

for name, scores in [("lead (first words)", LEAD_SCORES), ("aligned reference (bound)", ALIGNED_SCORES)]:
    print(f"{name:34s} {scores['rouge1']:8.4f} {scores['rouge2']:8.4f} {scores['rougeL']:8.4f} {scores['words']:7.0f}")

print(f"\nused for the test summaries: eos penalty {BEST_PENALTY}, length scale {BEST_SCALE}")

# repetition: share of repeated word 4-grams inside the generated summaries
def repeated_4grams(text):
    w = words_of(text)
    grams = list(zip(w, w[1:], w[2:], w[3:]))
    return 1 - len(set(grams)) / len(grams) if grams else 0.0

dev_candidates = [assemble(d, BEST_SCALE) for d in gen_dev_docs]
print(f"repeated word 4-grams: generated {np.mean([repeated_4grams(c) for c in dev_candidates]):.1%} | "
      f"reference {np.mean([repeated_4grams(r) for r in references]):.1%}")
print(f"segments skipped by the model (empty part): {np.mean([p == '' for d in gen_dev_docs for p in d['parts']]):.1%}")

for d, c, r in list(zip(gen_dev_docs, dev_candidates, references))[:3]:
    print(f"\n---- dev document {d['id']} ({d['words']} words, {len(d['parts'])} segments) ----")
    print(f"generated part of segment 1: {d['parts'][0][:300]!r}")
    print(f"generated summary ({word_count(c)} words): {c[:600]!r}")
    print(f"reference ({word_count(r)} words): {r[:600]!r}")


# ------------------------------------------------------------
# Save (dev predictions + references in the submission format -> official evaluation script / BERTScore)
# ------------------------------------------------------------

with open(OUTPUT_DIR / "dev_predictions.json", "w", encoding="utf-8") as f:
    json.dump([{"id": d["id"], "summary": c} for d, c in zip(gen_dev_docs, dev_candidates)], f, ensure_ascii=False)
with open(OUTPUT_DIR / "dev_references.json", "w", encoding="utf-8") as f:
    json.dump([{"id": d["id"], "summary": d["summary"]} for d in gen_dev_docs], f, ensure_ascii=False)

results = {
    "run": RUN_NAME,
    "model": "KD student: Architecture-3 decoder (frozen) + LoRA, teacher Architecture-4 (frozen) + LoRA; segment-wise summarization",
    "technique": {"method": "Knowledge distillation (token level)", "alpha": KD_ALPHA, "temperature": KD_TEMPERATURE,
                  "teacher": "Architecture-4 + LoRA", "student": "Architecture-3 + LoRA", "lora_rank": PEFT_RANK, "lr": LR_PEFT},
    "knowledge_distillation": {
        "teacher": {"weights": TEACHER_SIZE, "dev_loss": teacher_dev_loss, "train_minutes": teacher_minutes, "history": teacher_history},
        "student": {"weights": STUDENT_SIZE, "dev_loss": student_dev_loss, "train_minutes": train_minutes},
    },
    "segments": {"segment_tokens": SEGMENT_TOKENS, "target_tokens": TARGET_TOKENS, "max_segments": MAX_SEGMENTS,
                 "empty_keep": EMPTY_KEEP},
    "generation": {"greedy": True, "max_new_tokens": MAX_NEW_TOKENS, "no_repeat_ngram_non_space_tokens": NO_REPEAT_NGRAM,
                   "eos_penalty": BEST_PENALTY, "length_scale": BEST_SCALE, "length_model": {"a": float(LENGTH_A), "b": float(LENGTH_B)},
                   "segments_per_second": n_segments_dev / max(gen_seconds, 1e-9)},
    "parameters_total": n_total, "parameters_trained": n_trained,
    "best_epoch": best_epoch, "train_minutes": train_minutes, "peak_gpu_gb": peak_gpu_gb,
    "history": history,
    "dev_docs": len(gen_dev_docs), "dev": DEV_SCORES, "dev_by_setting": scale_results,
    "dev_lead": LEAD_SCORES, "dev_aligned_reference": ALIGNED_SCORES,
}

print(f"\nTEACHER vs STUDENT (dev loss = cross-entropy of the reference summary tokens)")
print(f"  teacher Architecture-4: {TEACHER_SIZE:>12,} weights | dev loss {teacher_dev_loss:.4f}")
print(f"  student Architecture-3: {STUDENT_SIZE:>12,} weights | dev loss {student_dev_loss:.4f} "
      f"| {STUDENT_SIZE / TEACHER_SIZE:.0%} of the teacher's size")

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("\nSaved:", OUTPUT_DIR / "results.json", "| dev_predictions.json + dev_references.json (for the official evaluator)")


summary                             ROUGE-1  ROUGE-2  ROUGE-L   words
reference                                                         677
    480/1175 segments | 7s
    960/1175 segments | 15s
model, eos -0.0, length x 0.6        0.0000   0.0000   0.0000       0
model, eos -0.0, length x 0.8        0.0000   0.0000   0.0000       0
model, eos -0.0, length x 1.0        0.0000   0.0000   0.0000       0
model, eos -0.0, length x 1.25       0.0000   0.0000   0.0000       0
model, eos -0.0, length x None       0.0000   0.0000   0.0000       0
    (1175 segments in 18s = 65.2 segments/s)
    480/1175 segments | 12s
    960/1175 segments | 25s
model, eos -2.0, length x 0.6        0.0033   0.0016   0.0026       2
model, eos -2.0, length x 0.8        0.0033   0.0016   0.0026       2
model, eos -2.0, length x 1.0        0.0033   0.0016   0.0026       2
model, eos -2.0, length x 1.25       0.0033   0.0016   0.0026       2
model, eos -2.0, length x None       0.0033   0.0016   0.0026       2
    

In [14]:
# ============================================================
# CELL 14 — SUMMARIZE THE TEST DOCUMENTS WITH THE STUDENT + WRITE prediction.json + submission.zip
#   format: [{"id": "doc_001", "summary": "..."}, ...]  (one summary for every test document)
# ============================================================

n_segments_test, test_seconds = generate_parts(model, test_docs, BEST_PENALTY)
print(f"generated {n_segments_test} segments of {len(test_docs)} test documents in {test_seconds / 60:.1f} min")

submission = []
for d in test_docs:
    summary = assemble(d, BEST_SCALE)
    if not summary.strip():                                  # never submit an empty summary
        summary = " ".join(d["document"].split()[:int(expected_summary_words(d["document"]))])
    submission.append({"id": d["id"], "summary": summary})


# ------------------------------------------------------------
# Format checks
# ------------------------------------------------------------

assert len(submission) == len(test_docs)
assert [s["id"] for s in submission] == [d["id"] for d in test_docs]
assert all(isinstance(s["summary"], str) and s["summary"].strip() for s in submission)

lengths = np.array([word_count(s["summary"]) for s in submission])
print(f"✅ {len(submission)} summaries | words: median {np.median(lengths):.0f}, min {lengths.min()}, max {lengths.max()}")

PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False)

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("submission.zip:", zf.namelist())

results["test_generation_minutes"] = test_seconds / 60
with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Saved:", ZIP_FILE)


    480/2187 segments | 88s
    960/2187 segments | 189s
    1440/2187 segments | 301s
    1920/2187 segments | 431s
generated 2187 segments of 100 test documents in 9.0 min
✅ 100 summaries | words: median 516, min 169, max 2430
submission.zip: ['prediction.json']
Saved: /kaggle/working/p6_kd_arch4_to_arch3/submission.zip
